# Is the radiomic branch what limits the physics layer under brightness shifts? (Kaggle)
The physics layer's *image* output is almost exactly invariant to darkening (a check on real patches gave a difference of 1.6 grey levels out of 255 at brightness x0.5),
yet the full model lost accuracy at brightness x0.5 (0.51). The radiomic vector is computed from the shifted raw image and is not invariant. This run **zeroes the radiomic input**
(training and test) and re-trains four models with seed 43 and the same folds and test conditions as before:
A_baseline, H_stainaug, C_stainOD+consist and N_reinhard. Compare with the earlier results with radiomics. Needs both Input datasets, GPU T4 x2 and Internet On. Expect roughly 1.5-2 h.

## 1. GPU check

In [ ]:
import os, subprocess, torch
print("torch", torch.__version__, "| GPU available:", torch.cuda.is_available())
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "No GPU: Session options > Accelerator > GPU T4 x2, then re-run"

## 2. Setup: find files, add the normaliser, patch the training code, unpack patches

In [ ]:
import os, shutil, zipfile, time, re
INPUT, WORK, CODE, TMP = "/kaggle/input", "/kaggle/working", "/kaggle/working/code", "/kaggle/temp"
os.makedirs(CODE, exist_ok=True); os.makedirs(TMP, exist_ok=True)

def find_file(name):
    stem, ext = os.path.splitext(name)
    pat = re.compile(rf"^{re.escape(stem)}( \((\d+)\))?{re.escape(ext)}$")
    hits = []
    for dp, _, fs in os.walk(INPUT):
        for f in fs:
            m = pat.match(f)
            if m:
                hits.append((int(m.group(2) or 0), os.path.join(dp, f)))
    if not hits:
        raise FileNotFoundError(f"{name} not found under {INPUT} - did you add both datasets as Input?")
    return max(hits)[1]

for f in ["ovarian_pipeline.py", "novelty_utils.py", "novel_model.py",
          "histo_utils.py", "histo_model.py", "test_histo_utils.py"]:
    src = find_file(f)
    shutil.copy(src, f"{CODE}/{f}")
    print("code:", os.path.basename(src), "->", f)

In [ ]:
STAIN_NORM_SRC = r'''"""
stain_norm.py - classical H&E stain normalisation baselines (numpy + OpenCV only, no extra packages).

  ReinhardNorm : colour transfer in Lab space (Reinhard et al., 2001). Target = mean of the per-image
                 Lab means / stds of a sample of TRAINING patches (so nothing is learned from test slides).
  MacenkoNorm  : stain-vector estimation by SVD + angle percentiles, concentration rescaling and
                 re-synthesis with a fixed reference H&E matrix (Macenko et al., 2009).

Both are applied per image (no slide-level information), to uint8 RGB arrays of shape (N,H,W,3).
If an image cannot be normalised (too little tissue, degenerate stain estimate) it is returned unchanged
and counted in `.n_fallback`, so the fallback rate can be reported.
"""
import cv2
import numpy as np

# Reference stain matrix and maximum concentrations used by the common Macenko implementations
HE_REF = np.array([[0.5626, 0.2159],
                   [0.7201, 0.8012],
                   [0.4062, 0.5581]], dtype=np.float64)
MAXC_REF = np.array([1.9705, 1.0308], dtype=np.float64)


class ReinhardNorm:
    name = "reinhard"

    def __init__(self, target_mean=None, target_std=None, bg_l=230):
        self.mu_t, self.sd_t, self.bg_l = target_mean, target_std, bg_l
        self.n_fallback = 0
        self.n_total = 0

    def _stats(self, lab):
        L = lab[..., 0]
        m = L < self.bg_l                               # drop (near-)white background
        px = lab[m] if m.sum() > 100 else lab.reshape(-1, 3)
        return px.mean(0), px.std(0) + 1e-6

    def fit(self, X, n=400, seed=0):
        idx = np.random.default_rng(seed).choice(len(X), min(n, len(X)), replace=False)
        st = [self._stats(cv2.cvtColor(X[i], cv2.COLOR_RGB2LAB).astype(np.float32)) for i in idx]
        self.mu_t = np.mean([s[0] for s in st], 0)
        self.sd_t = np.mean([s[1] for s in st], 0)
        return self

    def one(self, img):
        self.n_total += 1
        lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB).astype(np.float32)
        mu, sd = self._stats(lab)
        out = (lab - mu) / sd * self.sd_t + self.mu_t
        out = np.clip(out, 0, 255).astype(np.uint8)
        return cv2.cvtColor(out, cv2.COLOR_LAB2RGB)

    def __call__(self, X):
        return np.stack([self.one(im) for im in X])


class MacenkoNorm:
    name = "macenko"

    def __init__(self, io=255, alpha=1.0, beta=0.15, he_ref=HE_REF, maxc_ref=MAXC_REF):
        self.io, self.alpha, self.beta = io, alpha, beta
        self.he_ref, self.maxc_ref = he_ref, maxc_ref
        self.n_fallback = 0
        self.n_total = 0

    def fit(self, X, **kw):                              # fixed reference, nothing to fit
        return self

    def one(self, img):
        self.n_total += 1
        h, w, _ = img.shape
        od = -np.log((img.reshape(-1, 3).astype(np.float64) + 1.0) / self.io)
        keep = ~(od < self.beta).any(1)                  # remove transparent pixels
        if keep.sum() < 200:
            self.n_fallback += 1
            return img
        odh = od[keep]
        try:
            _, v = np.linalg.eigh(np.cov(odh.T))
            plane = v[:, 1:3]                            # two largest eigenvectors
            if plane[0, 0] < 0:
                plane[:, 0] *= -1
            if plane[0, 1] < 0:
                plane[:, 1] *= -1
            phi = np.arctan2(odh @ plane[:, 1], odh @ plane[:, 0])
            lo, hi = np.percentile(phi, self.alpha), np.percentile(phi, 100 - self.alpha)
            v1 = plane @ np.array([np.cos(lo), np.sin(lo)])
            v2 = plane @ np.array([np.cos(hi), np.sin(hi)])
            he = np.stack([v1, v2], 1) if v1[0] > v2[0] else np.stack([v2, v1], 1)   # H first (larger red OD)
            c = np.linalg.lstsq(he, od.T, rcond=None)[0]                              # (2, N)
            maxc = np.percentile(c, 99, axis=1)
            if not np.all(np.isfinite(maxc)) or (maxc < 1e-3).any() or not np.all(np.isfinite(he)):
                raise ValueError("degenerate stain estimate")
        except (np.linalg.LinAlgError, ValueError):
            self.n_fallback += 1
            return img
        c = c / maxc[:, None] * self.maxc_ref[:, None]
        out = self.io * np.exp(-self.he_ref @ c)
        return np.clip(out.T.reshape(h, w, 3), 0, 255).astype(np.uint8)

    def __call__(self, X):
        return np.stack([self.one(im) for im in X])


def make_normaliser(name, X_train):
    """name in {'reinhard','macenko'}; X_train used only to set the Reinhard target statistics."""
    n = {"reinhard": ReinhardNorm, "macenko": MacenkoNorm}[name]()
    return n.fit(X_train)
'''
open(f"{CODE}/stain_norm.py", "w").write(STAIN_NORM_SRC)
print("stain_norm.py written", len(STAIN_NORM_SRC), "chars")

In [ ]:
def apply_norm_patch(path):
    txt = open(path).read()
    if "stain_norm" in txt:
        return "already patched"
    def rep(old, new):
        nonlocal txt
        assert txt.count(old) == 1, f"patch anchor not found exactly once: {old[:60]!r}"
        txt = txt.replace(old, new)
    rep("import histo_utils as hu\n", "import histo_utils as hu\nimport stain_norm\n")
    anchor = '    "H_stainaug":        dict(physics=False, fusion="concat", consist=False, cjit=False, saug=True),\n'
    rep(anchor, anchor +
        '    "N_reinhard":        dict(physics=False, fusion="concat", consist=False, cjit=False, norm="reinhard"),\n'
        '    "N_macenko":         dict(physics=False, fusion="concat", consist=False, cjit=False, norm="macenko"),\n')
    rep("    def predict(self, model, X, R):\n        return self.predict_float(",
        "    def predict(self, model, X, R):\n"
        "        norm = getattr(model, \"normaliser\", None)\n"
        "        if norm is not None:                      # classical stain normalisation, applied to every image\n"
        "            X = norm(X)\n"
        "        return self.predict_float(")
    model_line = "        model = HistoNet(tr[\"R\"].shape[1], K, cfg[\"physics\"], cfg[\"fusion\"], not a.no_pretrained).to(dev)\n"
    rep(model_line, model_line +
        "        norm = stain_norm.make_normaliser(cfg[\"norm\"], tr[\"X\"]) if cfg.get(\"norm\") else None\n"
        "        model.normaliser = norm\n"
        "        if norm is not None:\n"
        "            tr = dict(tr, X=norm(tr[\"X\"]))\n"
        "            print(f\"      {cfg['norm']}: train images normalised, fallback {norm.n_fallback}/{norm.n_total}\", flush=True)\n")
    rep("    def hook(fold, models, d):\n",
        "    def hook(fold, models, d):\n"
        "        for _v, _m in models.items():\n"
        "            _n = getattr(_m, \"normaliser\", None)\n"
        "            if _n is not None:\n"
        "                print(f\"    [{_v}] normaliser fallback so far: {_n.n_fallback}/{_n.n_total} images\", flush=True)\n")
    open(path, "w").write(txt)
    return "patched"

print("histo_model.py:", apply_norm_patch(f"{CODE}/histo_model.py"))

# "wb<s>" colour-cast test (same patch as in the seed-43 notebook); skipped if already present
p = f"{CODE}/histo_utils.py"
txt = open(p).read()
old = '''    elif kind == "wb":
        out = img.astype(np.float32) * np.array([1.0, 0.9, 0.8], np.float32)'''
new = '''    elif kind.startswith("wb"):
        s = float(kind[2:]) if len(kind) > 2 else 0.1
        out = img.astype(np.float32) * np.array([1.0, 1 - s, 1 - 2 * s], np.float32)'''
if 'kind.startswith("wb")' not in txt:
    assert old in txt, "histo_utils.py differs from the expected version"
    open(p, "w").write(txt.replace(old, new))
print("wb patch ok")

In [ ]:
def apply_norad_patch(utils_path, model_path):
    out = []
    def patch(path, old, new, tag):
        txt = open(path).read()
        if tag in txt:
            out.append(f"{path.split('/')[-1]}: {tag} already present"); return
        assert txt.count(old) == 1, f"anchor not found exactly once in {path}: {old[:50]!r}"
        open(path, "w").write(txt.replace(old, new)); out.append(f"{path.split('/')[-1]}: patched")
    patch(utils_path, "                rr = radiomics_rgb(Xi, workers)\n",
          "                rr = (np.zeros((len(Xi), rad_all.shape[1]), np.float32) if os.environ.get(\"NO_RADIOMICS\") == \"1\"  # NO_RADIOMICS\n"
          "                      else radiomics_rgb(Xi, workers))\n", "# NO_RADIOMICS")
    patch(model_path, "import json\n", "import json\nimport os\n", "import os\n")
    patch(model_path, "    rad_all = hu.cached_radiomics(data[\"X\"], out / \"radiomics_cache.npz\", args.workers)\n",
          "    rad_all = hu.cached_radiomics(data[\"X\"], out / \"radiomics_cache.npz\", args.workers)\n"
          "    if os.environ.get(\"NO_RADIOMICS\") == \"1\":                      # NO_RADIOMICS\n"
          "        rad_all = np.zeros_like(rad_all)\n"
          "        print(\"radiomic branch disabled: all-zero radiomic input\", flush=True)\n", "# NO_RADIOMICS")
    return out

for line in apply_norad_patch(f"{CODE}/histo_utils.py", f"{CODE}/histo_model.py"):
    print(line)

In [ ]:
t = time.time()
UBC = f"{TMP}/ubc"
zips = [os.path.join(dp, f) for dp, _, fs in os.walk(INPUT) for f in fs if f == "ubc_patches.zip"]
if zips:
    with zipfile.ZipFile(zips[0]) as z:
        z.extractall(TMP)
else:
    man = find_file("patch_manifest.csv")
    shutil.copytree(os.path.dirname(os.path.dirname(man)), UBC, dirs_exist_ok=True)
assert os.path.exists(f"{UBC}/patches/patch_manifest.csv"), f"unexpected layout under {UBC}"
print(f"patches ready: {sum(len(fs) for _, _, fs in os.walk(UBC))} files in {time.time() - t:.0f}s")
os.chdir(CODE)

## 3. Smoke test with the radiomic branch off (a few minutes). Expect `radiomic branch disabled` and two small result tables

In [ ]:
os.environ["NO_RADIOMICS"] = "1"
!python histo_model.py --ubc-dir {UBC} --max-patches-per-slide 4 --folds 3 --epochs 2 --variants A_baseline C_stainOD+consist --workers 2 --dl-workers 2 --no-xai --out /kaggle/temp/check_norad 2>&1 | tail -25

## 4. Main run (seed 43, radiomics off). If the session stops, re-run this cell: `--resume` continues

In [ ]:
os.environ["NO_RADIOMICS"] = "1"
OUT = f"{WORK}/results_seed43_norad"
!python histo_model.py --ubc-dir {UBC} --max-patches-per-slide 16 --folds 5 --epochs 10 --seed 43 --variants A_baseline H_stainaug C_stainOD+consist N_reinhard --conditions clean clean_sub bright0.8 bright0.65 bright0.5 wb wb0.2 hscale1.3 hscale2.0 escale0.7 escale0.4 jpeg30 noise --workers 2 --dl-workers 2 --no-xai --resume --out {OUT}

## 5. Tables and zip (download `results_seed43_norad.zip` from the Output panel and attach it to the chat)

In [ ]:
print(open(f"{OUT}/tables.md", encoding="utf-8").read())
shutil.make_archive(f"{WORK}/results_seed43_norad", "zip", WORK, "results_seed43_norad")
print("zip MB:", round(os.path.getsize(f"{WORK}/results_seed43_norad.zip") / 1e6, 1))